In [1]:
import soccerdata as sd

import numpy as np
import pandas as pd

[11/25/25 17:43:08] INFO     No custom team name replacements found. You can configure these in       ]8;id=162152;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=612836;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site-packages/soccerdata/_config.py#91\91]8;;\
                             /Users/n/soccerdata/config/teamname_replacements.json.                                

                    INFO     Custom league dict loaded from                                          ]8;id=938651;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site-packages/soccerdata/_config.py\_config.py]8;;\:]8;id=109851;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site-packages/soccerdata/_config.py#195\195]8;;\
                             /Users/n/soccerdata/config/league_dict.json.                                          

In [2]:
sd.FBref.available_leagues()

['Big 5 European Leagues Combined',
 'ENG-Premier League',
 'ESP-La Liga',
 'FRA-Ligue 1',
 'GER-Bundesliga',
 'INT-European Championship',
 "INT-Women's World Cup",
 'INT-World Cup',
 'ITA-Serie A',
 'NED-Eredivisie',
 'NWSL']

In [3]:
fbref = sd.FBref(leagues="NWSL", seasons=2025)
# print(fbref.__doc__)

[11/25/25 17:43:10] INFO     Saving cached data to /Users/n/soccerdata/data/FBref                    ]8;id=235218;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site-packages/soccerdata/_common.py\_common.py]8;;\:]8;id=310689;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site-packages/soccerdata/_common.py#263\263]8;;\

In [6]:
cols_to_drop = [
    ("league", ""),
    ("season", ""),
    ("team", ""),
    ("player", ""),
    ("nation", ""),
    ("pos", ""),
    ("age", ""),
    ("born", ""),
    ("90s", ""),
]

player_cols = [("team", ""), ("player", "")]


def highlight_top10(s, props=""):
    return np.where(s <= 10, props, "")


def highlight_fbref_player_season(stat_type="shooting", topn=10, hide_non_qual=True):

    player_stats = fbref.read_player_season_stats(stat_type=stat_type).reset_index()

    if (("90s", "") in player_stats.columns) and hide_non_qual:

        max_matches = player_stats.loc[:, ("90s", "")].max()

        player_stats = player_stats.loc[
            player_stats.loc[:, ("90s", "")] >= max_matches / 4, :
        ]

    if stat_type == "misc":

        hide_aerial_rate = (
            player_stats.loc[
                :,
                [
                    ("Aerial Duels", "Won"),
                    ("Aerial Duels", "Lost"),
                ],
            ].sum(axis=1)
            < 15
        )

        player_stats.loc[hide_aerial_rate, ("Aerial Duels", "Won%")] = None

    if stat_type == "defense":
        hide_challenge_rate = player_stats.loc[:, ("Challenges", "Att")].sum() < 15

        player_stats.loc[hide_challenge_rate, ("Challenges", "Tkl%")] = None

    num_players = player_stats.shape[0]

    player_stats_numeric = player_stats.drop(columns=cols_to_drop, errors="ignore")

    player_stats_ranked = player_stats_numeric.rank(method="min", ascending=False)

    player_stats_order = player_stats_ranked.sum(axis=1).argsort()

    player_stats_ranked_full = pd.concat(
        (
            player_stats.loc[:, player_cols],
            player_stats_ranked,
        ),
        axis=1,
    ).iloc[player_stats_order, :]

    return (
        player_stats_ranked_full.loc[
            player_stats_ranked_full.loc[:, ("team", "")] == "Louisville", :
        ]
        .style.format(
            precision=0,
        )
        .highlight_between(
            subset=player_stats_numeric.columns,
            left=0,
            right=topn,
            props="background-color: #c9b3f7; color: black;",
        )
        .highlight_between(
            subset=player_stats_numeric.columns,
            left=num_players - topn,
            right=num_players,
            color="#800000",
        )
        .set_table_styles(
            [
                {
                    "selector": "th.col_heading",
                    "props": "border-left: 1px solid white; text-align: center;",
                }
            ]
        )
        .set_caption(
            f"Rank of Racing Louisville players across NWSL for 2025 Season: {stat_type.replace('_', ' ').title()}"
        )
        .hide()
    )

In [8]:
highlight_fbref_player_season(stat_type="standard", hide_non_qual=False)

[11/25/25 17:45:07] WARNING  /opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site- ]8;id=552883;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=156102;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py#110\110]8;;\
                             packages/soccerdata/fbref.py:165: FutureWarning: The behavior of                      
                             DataFrame concatenation with empty or all-NA entries is deprecated. In                
                             a future version, this will no longer exclude empty or all-NA columns                 
                             when determining the result dtypes. To retain the old behavior,                       
                             exclude the relevant entries before the concat operation.                             
                               pd.concat(dfs)                                                                      
                                                                                                                   

In [6]:
highlight_fbref_player_season(stat_type="shooting", hide_non_qual=False)

                    WARNING  /opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site- ]8;id=5467;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=303641;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py#110\110]8;;\
                             packages/soccerdata/fbref.py:165: FutureWarning: The behavior of                      
                             DataFrame concatenation with empty or all-NA entries is deprecated. In                
                             a future version, this will no longer exclude empty or all-NA columns                 
                             when determining the result dtypes. To retain the old behavior,                       
                             exclude the relevant entries before the concat operation.                             
                               pd.concat(dfs)                                                                      
                                                                                                                   

In [7]:
highlight_fbref_player_season(stat_type="passing")

                    WARNING  /opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site- ]8;id=576171;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=974816;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py#110\110]8;;\
                             packages/soccerdata/fbref.py:165: FutureWarning: The behavior of                      
                             DataFrame concatenation with empty or all-NA entries is deprecated. In                
                             a future version, this will no longer exclude empty or all-NA columns                 
                             when determining the result dtypes. To retain the old behavior,                       
                             exclude the relevant entries before the concat operation.                             
                               pd.concat(dfs)                                                                      
                                                                                                                   

In [8]:
highlight_fbref_player_season(stat_type="passing_types")

                    WARNING  /opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site- ]8;id=14926;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=942377;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py#110\110]8;;\
                             packages/soccerdata/fbref.py:165: FutureWarning: The behavior of                      
                             DataFrame concatenation with empty or all-NA entries is deprecated. In                
                             a future version, this will no longer exclude empty or all-NA columns                 
                             when determining the result dtypes. To retain the old behavior,                       
                             exclude the relevant entries before the concat operation.                             
                               pd.concat(dfs)                                                                      
                                                                                                                   

In [9]:
highlight_fbref_player_season(stat_type="goal_shot_creation")

                    WARNING  /opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site- ]8;id=324235;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=575990;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py#110\110]8;;\
                             packages/soccerdata/fbref.py:165: FutureWarning: The behavior of                      
                             DataFrame concatenation with empty or all-NA entries is deprecated. In                
                             a future version, this will no longer exclude empty or all-NA columns                 
                             when determining the result dtypes. To retain the old behavior,                       
                             exclude the relevant entries before the concat operation.                             
                               pd.concat(dfs)                                                                      
                                                                                                                   

In [10]:
highlight_fbref_player_season(stat_type="defense")

                    WARNING  /opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site- ]8;id=873781;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=9490;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py#110\110]8;;\
                             packages/soccerdata/fbref.py:165: FutureWarning: The behavior of                      
                             DataFrame concatenation with empty or all-NA entries is deprecated. In                
                             a future version, this will no longer exclude empty or all-NA columns                 
                             when determining the result dtypes. To retain the old behavior,                       
                             exclude the relevant entries before the concat operation.                             
                               pd.concat(dfs)                                                                      
                                                                                                                   

In [11]:
highlight_fbref_player_season(stat_type="possession")

[11/03/25 10:22:57] WARNING  /opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site- ]8;id=87393;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=883795;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py#110\110]8;;\
                             packages/soccerdata/fbref.py:165: FutureWarning: The behavior of                      
                             DataFrame concatenation with empty or all-NA entries is deprecated. In                
                             a future version, this will no longer exclude empty or all-NA columns                 
                             when determining the result dtypes. To retain the old behavior,                       
                             exclude the relevant entries before the concat operation.                             
                               pd.concat(dfs)                                                                      
                                                                                                                   

In [12]:
highlight_fbref_player_season(stat_type="playing_time")

                    WARNING  /opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site- ]8;id=78980;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=623345;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py#110\110]8;;\
                             packages/soccerdata/fbref.py:165: FutureWarning: The behavior of                      
                             DataFrame concatenation with empty or all-NA entries is deprecated. In                
                             a future version, this will no longer exclude empty or all-NA columns                 
                             when determining the result dtypes. To retain the old behavior,                       
                             exclude the relevant entries before the concat operation.                             
                               pd.concat(dfs)                                                                      
                                                                                                                   

In [13]:
highlight_fbref_player_season(stat_type="misc")

                    WARNING  /opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site- ]8;id=8385;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=516476;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py#110\110]8;;\
                             packages/soccerdata/fbref.py:165: FutureWarning: The behavior of                      
                             DataFrame concatenation with empty or all-NA entries is deprecated. In                
                             a future version, this will no longer exclude empty or all-NA columns                 
                             when determining the result dtypes. To retain the old behavior,                       
                             exclude the relevant entries before the concat operation.                             
                               pd.concat(dfs)                                                                      
                                                                                                                   

In [14]:
highlight_fbref_player_season(stat_type="keeper", topn=3)

                    WARNING  /opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site- ]8;id=854589;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=644158;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py#110\110]8;;\
                             packages/soccerdata/fbref.py:165: FutureWarning: The behavior of                      
                             DataFrame concatenation with empty or all-NA entries is deprecated. In                
                             a future version, this will no longer exclude empty or all-NA columns                 
                             when determining the result dtypes. To retain the old behavior,                       
                             exclude the relevant entries before the concat operation.                             
                               pd.concat(dfs)                                                                      
                                                                                                                   

In [15]:
highlight_fbref_player_season(stat_type="keeper_adv", topn=3, hide_non_qual=False)

                    WARNING  /opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site- ]8;id=846713;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=869184;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py#110\110]8;;\
                             packages/soccerdata/fbref.py:165: FutureWarning: The behavior of                      
                             DataFrame concatenation with empty or all-NA entries is deprecated. In                
                             a future version, this will no longer exclude empty or all-NA columns                 
                             when determining the result dtypes. To retain the old behavior,                       
                             exclude the relevant entries before the concat operation.                             
                               pd.concat(dfs)                                                                      
                                                                                                                   

In [16]:
def highlight_fbref_team_season(stat_type="standard", topn=3):

    cols_to_drop = [("league", ""), ("season", ""), ("team", ""), ("url", "")]

    team_stats = fbref.read_team_season_stats(stat_type=stat_type).reset_index()

    num_teams = team_stats.shape[0]

    team_stats_numeric = team_stats.drop(columns=cols_to_drop, errors="ignore")

    team_stats_ranked = pd.concat(
        (
            team_stats.loc[:, ("team", "")],
            team_stats_numeric.rank(method="first", ascending=False),
        ),
        axis=1,
    )

    return (
        team_stats_ranked.style.format(
            precision=0,
        )
        .highlight_between(
            subset=(7, team_stats_numeric.columns),
            left=0,
            right=topn,
            props="background-color: #c9b3f7; color: black;",
        )
        .highlight_between(
            subset=(7, team_stats_numeric.columns),
            left=num_teams - topn,
            right=num_teams,
            color="#800000",
        )
        .set_table_styles(
            [
                {
                    "selector": "th.col_heading",
                    "props": "border-left: 1px solid white; text-align: center;",
                }
            ]
        )
        .set_caption(
            f"Rank of Racing Louisville for NWSL 2025 Season: {stat_type.replace('_', ' ').title()}"
        )
        .hide()
    )

In [17]:
highlight_fbref_team_season()

                    WARNING  /opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site- ]8;id=638965;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=99304;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py#110\110]8;;\
                             packages/soccerdata/fbref.py:165: FutureWarning: The behavior of                      
                             DataFrame concatenation with empty or all-NA entries is deprecated. In                
                             a future version, this will no longer exclude empty or all-NA columns                 
                             when determining the result dtypes. To retain the old behavior,                       
                             exclude the relevant entries before the concat operation.                             
                               pd.concat(dfs)                                                                      
                                                                                                                   

In [18]:
highlight_fbref_team_season(stat_type="keeper")

                    WARNING  /opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site- ]8;id=73297;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=153900;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py#110\110]8;;\
                             packages/soccerdata/fbref.py:165: FutureWarning: The behavior of                      
                             DataFrame concatenation with empty or all-NA entries is deprecated. In                
                             a future version, this will no longer exclude empty or all-NA columns                 
                             when determining the result dtypes. To retain the old behavior,                       
                             exclude the relevant entries before the concat operation.                             
                               pd.concat(dfs)                                                                      
                                                                                                                   

In [19]:
highlight_fbref_team_season(stat_type="keeper_adv")

                    WARNING  /opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site- ]8;id=386396;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=649665;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py#110\110]8;;\
                             packages/soccerdata/fbref.py:165: FutureWarning: The behavior of                      
                             DataFrame concatenation with empty or all-NA entries is deprecated. In                
                             a future version, this will no longer exclude empty or all-NA columns                 
                             when determining the result dtypes. To retain the old behavior,                       
                             exclude the relevant entries before the concat operation.                             
                               pd.concat(dfs)                                                                      
                                                                                                                   

In [20]:
highlight_fbref_team_season(stat_type="shooting")

                    WARNING  /opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site- ]8;id=702472;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=230823;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py#110\110]8;;\
                             packages/soccerdata/fbref.py:165: FutureWarning: The behavior of                      
                             DataFrame concatenation with empty or all-NA entries is deprecated. In                
                             a future version, this will no longer exclude empty or all-NA columns                 
                             when determining the result dtypes. To retain the old behavior,                       
                             exclude the relevant entries before the concat operation.                             
                               pd.concat(dfs)                                                                      
                                                                                                                   

In [21]:
highlight_fbref_team_season(stat_type="passing")

                    WARNING  /opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site- ]8;id=217590;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=329496;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py#110\110]8;;\
                             packages/soccerdata/fbref.py:165: FutureWarning: The behavior of                      
                             DataFrame concatenation with empty or all-NA entries is deprecated. In                
                             a future version, this will no longer exclude empty or all-NA columns                 
                             when determining the result dtypes. To retain the old behavior,                       
                             exclude the relevant entries before the concat operation.                             
                               pd.concat(dfs)                                                                      
                                                                                                                   

In [22]:
highlight_fbref_team_season(stat_type="passing_types")

                    WARNING  /opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site- ]8;id=596071;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=719013;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py#110\110]8;;\
                             packages/soccerdata/fbref.py:165: FutureWarning: The behavior of                      
                             DataFrame concatenation with empty or all-NA entries is deprecated. In                
                             a future version, this will no longer exclude empty or all-NA columns                 
                             when determining the result dtypes. To retain the old behavior,                       
                             exclude the relevant entries before the concat operation.                             
                               pd.concat(dfs)                                                                      
                                                                                                                   

In [23]:
highlight_fbref_team_season(stat_type="goal_shot_creation")

                    WARNING  /opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site- ]8;id=697095;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=48411;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py#110\110]8;;\
                             packages/soccerdata/fbref.py:165: FutureWarning: The behavior of                      
                             DataFrame concatenation with empty or all-NA entries is deprecated. In                
                             a future version, this will no longer exclude empty or all-NA columns                 
                             when determining the result dtypes. To retain the old behavior,                       
                             exclude the relevant entries before the concat operation.                             
                               pd.concat(dfs)                                                                      
                                                                                                                   

In [24]:
highlight_fbref_team_season(stat_type="defense")

[11/03/25 10:22:58] WARNING  /opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site- ]8;id=565214;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=318627;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py#110\110]8;;\
                             packages/soccerdata/fbref.py:165: FutureWarning: The behavior of                      
                             DataFrame concatenation with empty or all-NA entries is deprecated. In                
                             a future version, this will no longer exclude empty or all-NA columns                 
                             when determining the result dtypes. To retain the old behavior,                       
                             exclude the relevant entries before the concat operation.                             
                               pd.concat(dfs)                                                                      
                                                                                                                   

In [25]:
highlight_fbref_team_season(stat_type="possession")

                    WARNING  /opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site- ]8;id=81505;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=107645;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py#110\110]8;;\
                             packages/soccerdata/fbref.py:165: FutureWarning: The behavior of                      
                             DataFrame concatenation with empty or all-NA entries is deprecated. In                
                             a future version, this will no longer exclude empty or all-NA columns                 
                             when determining the result dtypes. To retain the old behavior,                       
                             exclude the relevant entries before the concat operation.                             
                               pd.concat(dfs)                                                                      
                                                                                                                   

In [26]:
highlight_fbref_team_season(stat_type="playing_time")

                    WARNING  /opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site- ]8;id=758684;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=999638;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py#110\110]8;;\
                             packages/soccerdata/fbref.py:165: FutureWarning: The behavior of                      
                             DataFrame concatenation with empty or all-NA entries is deprecated. In                
                             a future version, this will no longer exclude empty or all-NA columns                 
                             when determining the result dtypes. To retain the old behavior,                       
                             exclude the relevant entries before the concat operation.                             
                               pd.concat(dfs)                                                                      
                                                                                                                   

In [27]:
highlight_fbref_team_season(stat_type="misc")

                    WARNING  /opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/site- ]8;id=333295;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py\warnings.py]8;;\:]8;id=834322;file:///opt/homebrew/Caskroom/miniconda/base/envs/soccer/lib/python3.11/warnings.py#110\110]8;;\
                             packages/soccerdata/fbref.py:165: FutureWarning: The behavior of                      
                             DataFrame concatenation with empty or all-NA entries is deprecated. In                
                             a future version, this will no longer exclude empty or all-NA columns                 
                             when determining the result dtypes. To retain the old behavior,                       
                             exclude the relevant entries before the concat operation.                             
                               pd.concat(dfs)                                                                      
                                                                                                                   